# **Robust Estimation: RANSAC vs. USAC/MAGSAC++**

## **Objectives:**
1. Swap notebook 27's `cv2.RANSAC` homography fit for OpenCV's modern USAC framework (`cv2.USAC_MAGSAC` and friends) — one function argument, no external model
2. Rerun on the exact same SIFT matches, both notebook 27's clean ratio-tested set and the noisy raw candidate set it filtered out
3. Measure inlier count, reprojection error, *and* runtime — find out which of those actually changes, and by how much

> **Assets.** No new files — reuses notebook 27's exact `assets/images/box.png` / `box_in_scene.png`. USAC has shipped in OpenCV since 4.5.1 (so it's on both this repo's pinned `4.10.0.84` and `.venv5x`'s `5.0.0.93`) — this isn't a 5.x-exclusive feature, just one the repo hadn't used yet.

In [1]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Define our imshow function
def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

## **RANSAC's actual bottleneck**

Classical RANSAC (notebook 27's `cv2.RANSAC`) repeats a simple loop: randomly sample the minimum points needed for a homography, fit it, count how many of *all* the matches agree with it within a fixed pixel threshold, and keep whichever random sample scored best — repeating enough times to be statistically confident the best sample seen was actually a clean one. That last part is the bottleneck: with more noise in the data, RANSAC needs *more* random samples to be confident it got a clean one by chance, and it re-scores *every* match against *every* candidate model.

**MAGSAC++** ("marginalizing sample consensus", accessed via `cv2.USAC_MAGSAC`) keeps the same random-sampling skeleton but replaces the expensive parts: a local optimization step refines promising models instead of relying on pure random luck, and — its signature idea — it never commits to one fixed inlier/outlier pixel threshold, marginalizing over a range of thresholds instead so one bad threshold choice can't sink the whole fit. Together these mean far fewer iterations are needed to reach the same confidence.

## **Rebuilding notebook 27's exact match sets**

Same SIFT keypoints, same `BFMatcher` + Lowe's ratio test — but this time keeping *both* outputs: the 80 matches that passed the ratio test, and all 604 raw nearest-neighbor candidates it filtered out of.

In [2]:
box = cv2.imread('assets/images/box.png')
scene = cv2.imread('assets/images/box_in_scene.png')
box_gray = cv2.cvtColor(box, cv2.COLOR_BGR2GRAY)
scene_gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp_box, des_box = sift.detectAndCompute(box_gray, None)
kp_scene, des_scene = sift.detectAndCompute(scene_gray, None)

bf = cv2.BFMatcher()
knn_matches = bf.knnMatch(des_box, des_scene, k=2)
raw_matches = [m for m, n in knn_matches]                                 # every candidate, unfiltered
good_matches = [m for m, n in knn_matches if m.distance < 0.75 * n.distance]  # notebook 27's exact ratio test

print(f'{len(raw_matches)} raw candidate matches -> {len(good_matches)} after Lowe\'s ratio test')

604 raw candidate matches -> 80 after Lowe's ratio test


## **Head-to-head: inliers, accuracy, and speed**

`cv2.findHomography`'s `method` argument is the only thing that changes between runs — everything else (the matches, the 5px reprojection threshold) stays identical to notebook 27, so any difference in the results below is real, not an artifact of a different setup.

In [3]:
import time

def eval_method(matches, method, name, reproj_thresh=5.0, n_runs=20):
    src = np.float32([kp_box[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp_scene[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    t0 = time.perf_counter()
    for _ in range(n_runs):
        H, mask = cv2.findHomography(src, dst, method, reproj_thresh)
    elapsed_ms = (time.perf_counter() - t0) / n_runs * 1000
    n_inliers = int(mask.sum())
    proj = cv2.perspectiveTransform(src, H)
    reproj_err = np.linalg.norm(proj - dst, axis=2).ravel()[mask.ravel().astype(bool)].mean()
    print(f'{name:16s}: {n_inliers:4d}/{len(matches):4d} inliers ({100 * n_inliers / len(matches):5.1f}%)  '
          f'mean reproj err={reproj_err:.3f}px  time={elapsed_ms:7.3f}ms')
    return H, mask

print('=== On the 80 ratio-tested matches (notebook 27\'s exact input) ===')
eval_method(good_matches, cv2.RANSAC, 'RANSAC')
eval_method(good_matches, cv2.USAC_MAGSAC, 'USAC_MAGSAC')

print()
print('=== On all 604 raw candidates — no ratio test, the noisy case ===')
eval_method(raw_matches, cv2.RANSAC, 'RANSAC')
eval_method(raw_matches, cv2.USAC_MAGSAC, 'USAC_MAGSAC')
eval_method(raw_matches, cv2.USAC_ACCURATE, 'USAC_ACCURATE')
eval_method(raw_matches, cv2.USAC_FAST, 'USAC_FAST')

=== On the 80 ratio-tested matches (notebook 27's exact input) ===
RANSAC          :   75/  80 inliers ( 93.8%)  mean reproj err=0.378px  time=  0.097ms
USAC_MAGSAC     :   75/  80 inliers ( 93.8%)  mean reproj err=0.383px  time=  0.078ms

=== On all 604 raw candidates — no ratio test, the noisy case ===


RANSAC          :   93/ 604 inliers ( 15.4%)  mean reproj err=0.502px  time= 14.130ms
USAC_MAGSAC     :   93/ 604 inliers ( 15.4%)  mean reproj err=0.504px  time=  0.241ms
USAC_ACCURATE   :   93/ 604 inliers ( 15.4%)  mean reproj err=0.522px  time=  0.736ms
USAC_FAST       :   93/ 604 inliers ( 15.4%)  mean reproj err=0.504px  time=  0.195ms


(array([[ 4.38062609e-01, -1.55087902e-01,  1.18664872e+02],
        [-3.81246260e-03,  4.14129964e-01,  1.61009145e+02],
        [-2.69215325e-04, -3.08254765e-04,  1.00000000e+00]]),
 array([[0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [1],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [

## **The result: identical accuracy, dramatically different speed**

On the already-clean 80 matches, every method lands on the same 75 inliers with sub-pixel reprojection error — there's barely any noise left for a smarter sampler to help with. The real difference shows up on the noisy 604-match set: every USAC variant finds the *exact same* 93 inliers, at essentially the *same* reprojection error, as classical RANSAC — but in roughly **20-70x less time**. Nothing about the answer changed; only how expensively RANSAC's blind random search had to work to find it.

That's a genuinely rare, uncomplicated result for this repo — most comparisons here trade one thing for another (notebook 35's sharper-but-lower-PSNR super-resolution, notebook 42's denser-but-less-accurate disparity filtering). This one doesn't: MAGSAC++ isn't a different trade-off, it's a strictly faster way to compute the same answer, which matters most exactly where notebook 27's ratio test *isn't* available to clean the data first — real-time matching against a large, noisy candidate set, frame after frame.

### A couple of directions to take this

- Rerun this exact comparison on notebook 28's `pano_left.jpg`/`pano_mid.jpg` matches — stitching's homography fit is the same kind of problem, on a different image pair.
- Sweep the reprojection threshold (`5.0` here) down to `1.0` and up to `15.0` for both methods — MAGSAC++'s threshold-marginalizing design is supposed to make it *less* sensitive to this choice than classical RANSAC; check whether that actually holds on this data.
- Re-detect with ORB at a much larger feature budget (`cv2.ORB_create(nfeatures=5000)`) instead of SIFT's ~600 keypoints — a noisier, larger candidate set is exactly where MAGSAC++'s speed advantage should widen further.